# 机器学习概述（下）Runner 类与波士顿房价案例

本节做两件事：

1. 抽象一个 **Runner** 类，把训练循环、评估、保存/加载封装起来——后续章节会持续复用并扩展它。
2. 跑通**波士顿房价回归**的端到端案例：数据清洗 → 数据集划分 → 特征归一化 → 训练 → 评估 → 预测。

## 1. Runner：一个最小训练骨架

PyTorch 的训练循环模板很固定：

```python
for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader:
        loss = loss_fn(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    eval_metric = evaluate(model, dev_loader)
```

我们把它包成 `Runner`，约定接口：`fit / evaluate / predict / save / load`。

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


class Runner:
    """训练 / 评估 / 保存 / 加载 的最小封装。

    后续章节会扩展：early stopping、metrics 记录、学习率调度等。
    """

    def __init__(self, model, optimizer, loss_fn, metric_fn=None):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.metric_fn = metric_fn      # 缺省时用 loss 当指标
        self.history = {'train_loss': [], 'dev_metric': []}

    def fit(self, train_loader, dev_loader=None, num_epochs=100, log_every=10):
        for epoch in range(num_epochs):
            self.model.train()
            running = 0.0
            for x, y in train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(x), y)
                loss.backward()
                self.optimizer.step()
                running += loss.item() * x.size(0)
            train_loss = running / len(train_loader.dataset)
            self.history['train_loss'].append(train_loss)

            if dev_loader is not None:
                dev_m = self.evaluate(dev_loader)
                self.history['dev_metric'].append(dev_m)
                if (epoch + 1) % log_every == 0:
                    print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}  dev_metric={dev_m:.4f}')
            elif (epoch + 1) % log_every == 0:
                print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}')

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total, n = 0.0, 0
        fn = self.metric_fn if self.metric_fn is not None else self.loss_fn
        for x, y in loader:
            total += fn(self.model(x), y).item() * x.size(0)
            n += x.size(0)
        return total / n

    @torch.no_grad()
    def predict(self, x):
        self.model.eval()
        return self.model(x)

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path))

## 2. 波士顿房价回归

本案例数据集为波士顿城郊 506 个样本，原 14 列。我们**去掉 `b` 列**（该列是按种族划分的统计变量，存在伦理问题；sklearn 自 1.0 起也已停止内置该数据），剩余 12 个特征预测房价 `medv`。

数据集放在 [`../dataset/boston_house_prices.csv`](../dataset/boston_house_prices.csv)。

### 2.1 加载与清洗：缺失值 / 异常值

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv('../dataset/boston_house_prices.csv')
data.columns = [c.upper() for c in data.columns]
data = data.drop(columns=['B'])         # 去掉伦理问题列
print('shape:', data.shape)
print('missing per column:\n', data.isna().sum().to_string())
data.head()

用四分位法把异常值截断到 $[Q_1 - 1.5\,\mathrm{IQR},\, Q_3 + 1.5\,\mathrm{IQR}]$ 区间。`CHAS` 是 0/1 类别，不做截断。

In [ ]:
def clip_outliers(df, exclude=()):
    df = df.copy()
    for col in df.columns:
        if col in exclude:
            continue
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[col] = df[col].clip(lo, hi)
    return df

data = clip_outliers(data, exclude={'CHAS'})

### 2.2 划分训练 / 测试集 + 特征归一化

**关键约定**：归一化的 `min/max`（或 `mean/std`）只能在训练集上拟合，再用同一组统计量去变换测试集——否则会发生测试集信息泄露。

In [ ]:
torch.manual_seed(0)

X = torch.tensor(data.drop(columns=['MEDV']).values, dtype=torch.float32)
y = torch.tensor(data['MEDV'].values, dtype=torch.float32).unsqueeze(1)

N = len(X)
perm = torch.randperm(N)
split = int(N * 0.8)
tr, te = perm[:split], perm[split:]
X_train, y_train = X[tr], y[tr]
X_test,  y_test  = X[te], y[te]

# 只在训练集上拟合 min/max
X_min, X_max = X_train.min(dim=0).values, X_train.max(dim=0).values
X_train = (X_train - X_min) / (X_max - X_min)
X_test  = (X_test  - X_min) / (X_max - X_min)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=32)
print(f'train: {len(X_train)}  test: {len(X_test)}  feature dim: {X_train.shape[1]}')

### 2.3 模型 + Runner + 训练

线性回归模型 = `nn.Linear(D, 1)`。用 MSE 作为损失也作为评估指标。

In [ ]:
torch.manual_seed(0)

input_size = X_train.shape[1]    # 12
model = nn.Linear(input_size, 1)
optimizer = optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

runner = Runner(model, optimizer, loss_fn)
runner.fit(train_loader, dev_loader=test_loader, num_epochs=200, log_every=40)

### 2.4 保存与加载

In [ ]:
save_path = './models/boston_linear.pt'
runner.save(save_path)

# 新建一个模型，从磁盘加载参数
model2 = nn.Linear(input_size, 1)
runner2 = Runner(model2, optim.Adam(model2.parameters()), loss_fn)
runner2.load(save_path)
print('test MSE (reloaded):', runner2.evaluate(test_loader))

### 2.5 检查学到的权重

在归一化的特征上，权重正负就能定性解释——负权重表示"该特征增大 → 房价下降"。

In [ ]:
feat_names = list(data.drop(columns=['MEDV']).columns)
weights = model.weight.detach().squeeze().tolist()
bias = model.bias.item()

for name, w in sorted(zip(feat_names, weights), key=lambda kv: kv[1]):
    print(f'{name:8s}  {w:+.4f}')
print(f'\nbias       {bias:+.4f}')

观察：`LSTAT`（低收入人群占比）与 `CRIM`（人均犯罪率）权重为负，`RM`（房间数）权重为正，与直觉一致。

### 2.6 单点预测

In [ ]:
x0, y0 = X_test[:1], y_test[:1]
print(f'true price:  {y0.item():.2f}')
print(f'pred price:  {runner.predict(x0).item():.2f}')

### 2.7 训练曲线

In [ ]:
plt.plot(runner.history['train_loss'], label='train loss')
plt.plot(runner.history['dev_metric'], label='test MSE')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 小结

- **Runner** 把训练循环、评估、模型 IO 封装成一个对象——后续章节会扩展（验证集、早停、metric 记录、学习率调度）。
- **特征归一化** 的统计量必须只在训练集上拟合，再去变换测试集。
- **`state_dict` 保存/加载** 是 PyTorch 的官方推荐做法（比直接 pickle 整个模型更稳）。
- 数据集和异常值处理：用 `pandas` 加载，IQR 截断处理异常值，去掉 `b` 列。